# WS1 — Empirical-Bayes Conditional Tables: The Transparent Baseline

**Workstream 1 of the pitch-sequencing rigor ladder.** This notebook fits and reads the
simplest honest model in the whole study: a set of **lookup tables**, one per state view,
each shrunk toward a well-estimated parent by empirical Bayes. It is deliberately the least
powerful model we will build — and that is the point. Everything fancier (a variable-order
Markov grammar in WS2, gradient-boosted trees in WS3, a GRU in WS6, offline RL in WS7) has to
*beat this* before its extra machinery has earned its keep.

## What this workstream asks

Two questions, in the plainest possible form:

1. **Selection.** Given the count, the handedness matchup, the pitcher, and the pitches
   already thrown this plate appearance, what is the probability distribution over the *next*
   pitch family? (A hierarchical Dirichlet-multinomial table.)
2. **Run value.** Given the same conditioning *and the pitch actually thrown*, what is the
   expected reward `E[R]` of the pitch, and how uncertain is that estimate? (A hierarchical
   normal partial-pooling table.)

We ask both across the nested **state views** `C ⊆ U, L1 ⊆ O` and watch what happens to the
answers — and to the *support* — as the conditioning key deepens.

## Its place in the rigor ladder

The project keeps three findings separate that the baseball-analytics literature routinely
conflates (SPEC §0, quoted verbatim):

> 1. **Selection structure** — prior pitches help predict *what is thrown next*.
> 2. **Predictive sequencing value** — prior pitches help predict the *outcome* of the
>    current pitch, after conditioning on the current pitch and game state.
> 3. **Prescriptive/causal value** — *changing* the sequence would improve outcomes.
>
> The first is easy; the second is hard; the third needs assumptions that public data
> cannot fully satisfy.

WS1 lives at the **bottom** of that ladder. It can speak to finding #1 (does ordered history
sharpen the next-pitch distribution?) and can *probe* finding #2 through its run-value tables,
but a flat conditional table is a blunt instrument for both — and its single most valuable
contribution is to make the **support problem** concrete and unavoidable: the reason we need
the rest of the ladder at all.

## Execution note

This notebook is a **scaffold**. Phase 2 executes it on the real Statcast decision table. A
single toggle, `DATA_MODE`, selects the world:

- `'synth_null'` — a simulated world with order-dependent *selection* habits but **no**
  ordered *outcome* effect (the correctness oracle's null world). Default.
- `'synth_positive'` — the same engine plus a **planted** ordered previous-transition effect
  on whiffs (the oracle's positive world).
- `'real'` — the real decision table built by `python -m pitchseq.build_table` (Phase 2).

Everything below runs top-to-bottom in every mode. The synthetic modes need no data files and
serve as the falsification checks; the real mode is the deliverable.

## How to read this notebook

Every code step is bracketed by plain-worded markdown: **before** each cell we say what will
happen and why; **after** each cell we say how to read what came out. Numbers that depend on
the real data are written as `{PLACEHOLDER}` in the companion `PAPER.md`; here they simply
appear when you run the cell. The **Results** section (§9) is *branched*: it inspects the
computed numbers and tells you which pre-written interpretation applies, so the conclusion is
ready the moment the cells finish.

## 1. Setup

We put the repository root on `sys.path` (so the `pitchseq` package and the
`workstreams.ws1_eb_tables` module import whether the notebook is launched from the repo root
or from `notebooks/`), load the shared study config, and set the two knobs that steer the
whole notebook: `DATA_MODE` (which world) and `SEED` (reproducibility). Nothing here touches
data yet.

In [ ]:
import sys
import json
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# --- locate the repository root (works from repo root or from notebooks/) ---
REPO_ROOT = Path.cwd()
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "pyproject.toml").exists() and (_p / "workstreams").is_dir():
        REPO_ROOT = _p
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# --- shared foundation (WS0) ---
from pitchseq.config import load_config
from pitchseq.splits import make_splits
from pitchseq.families import FAMILIES
from pitchseq.outcomes import OUTCOME1
from pitchseq.eval.baselines import build_baselines
from pitchseq.eval.predictions import ACTION_PROB_COLS
from pitchseq.eval.metrics import reliability_table
from pitchseq.eval.harness import evaluate_predictions, compare_views, slice_masks

# --- WS1 (this workstream) ---
from workstreams.ws1_eb_tables import model as ebm
from workstreams.ws1_eb_tables.run_ws1 import run_ws1

CONFIG = load_config()
SEED = int(CONFIG.get("seeds", {}).get("global", 20260713))

# The world this run analyses: 'synth_null' | 'synth_positive' | 'real'.
DATA_MODE = "synth_null"

# The feasible views for a pure table (SPEC §6 ladder order; OM is infeasible — see §7).
VIEWS = ["C", "U", "L1", "O"]

# Cluster-bootstrap replicates for the confidence intervals. Only affects CI width; lower it
# for a quick pass on the full validation season (SPEC §7 clusters by pitcher-game).
N_BOOT = 100

# Where the real decision table lives after `python -m pitchseq.build_table` (Phase 2).
REAL_TABLE_PATH = REPO_ROOT / "data" / "processed" / "decision_table.parquet"

print(f"repo root : {REPO_ROOT}")
print(f"DATA_MODE : {DATA_MODE}")
print(f"views     : {VIEWS}")
print(f"seed      : {SEED}")

### Plotting style (fixed, colorblind-safe view colours)

Every figure in this notebook uses the **Okabe–Ito** palette (safe for the common forms of
colour-vision deficiency) and a **fixed view → colour map** so that a given state view is
*always* the same colour, in every figure, throughout the notebook. We also strip the top and
right spines and drop gridlines by default: less ink, easier reading. This cell defines the
palette, the map, and two small helpers used everywhere below.

In [ ]:
# Okabe–Ito qualitative palette (colorblind-safe).
OKABE_ITO = {
    "orange":        "#E69F00",
    "sky_blue":      "#56B4E9",
    "bluish_green":  "#009E73",
    "yellow":        "#F0E442",
    "blue":          "#0072B2",
    "vermillion":    "#D55E00",
    "reddish_purple":"#CC79A7",
    "black":         "#000000",
}

# Fixed view -> colour. C is the context-only anchor; O is the fully ordered headline view.
VIEW_COLORS = {
    "C":  OKABE_ITO["blue"],
    "U":  OKABE_ITO["orange"],
    "L1": OKABE_ITO["bluish_green"],
    "O":  OKABE_ITO["vermillion"],
    "OM": OKABE_ITO["reddish_purple"],
}

# A neutral colour for the count-based reference baselines.
REF_COLOR = OKABE_ITO["black"]

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 110,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "figure.autolayout": True,
})


def style_axes(ax):
    '''Left+bottom spines only; no top/right. Returns the axis for chaining.'''
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    return ax


def new_fig(figsize=(7.2, 4.2)):
    '''One figure, one axis, pre-styled.'''
    fig, ax = plt.subplots(figsize=figsize)
    style_axes(ax)
    return fig, ax

## 2. Data and exploratory analysis

### The decision table, and why leakage discipline matters

Every workstream reads the same **canonical decision table** (SPEC §3): one row per pitch,
representing the *decision made immediately before that pitch is released*, keyed by
`(game_pk, at_bat_number, pitch_number)`. Its columns fall into groups:

- **Identity / keys** — `game_pk, at_bat_number, pitch_number, pitcher, batter, season,
  pa_id, row_id`.
- **Non-sequence context `X_t`** (this *is* the **C** view) — the count `balls, strikes`,
  `outs_when_up`, base state, inning, score, `stand / p_throws`, `pitcher_pitch_count`,
  `times_thru_order`, trailing repertoire mix and batter tendencies, park.
- **Within-PA history `H^PA_t`** — for each prior pitch this PA, a token: its `family`,
  batter-relative location, `release_speed` and movement, and an outcome token. These add the
  **U / L1 / O** views.
- **Labels (not features)** — the current pitch's own execution `Z_t` (velocity, movement,
  location — used only by an outcome model *conditioned on the pitch thrown*), the event-tree
  outcome `Y_t`, and the reward `R_t = -delta_run_exp`.

The **one non-negotiable discipline** is leakage (SPEC §0): a feature for the decision before
pitch *t* must be computable strictly *before pitch t is thrown*. Concretely, **the current
pitch's own velocity is not a feature.** It is an *execution* measurement — a consequence of
the decision, not an input to it. Using it would let the model "predict" the pitch from the
pitch. Prior pitches' velocities are fair game (they are known before the current decision);
the current one never is. The shared builders enforce this with a column-name audit; WS1
inherits that guarantee for free by reading only the audited table.

The next cell loads or synthesises the table for the chosen `DATA_MODE`, then applies the
locked **temporal split** (SPEC §7): train on 2021–2023, validate on 2024. Splits are by
season, never by random row — pitches in the same PA/game are far too dependent for a random
split to be honest.

In [ ]:
def load_decision_table(mode: str, config: dict, seed: int):
    '''Return (decision_table, truth_meta) for the chosen world.'''
    if mode == "real":
        if not REAL_TABLE_PATH.exists():
            raise FileNotFoundError(
                f"real decision table not found at {REAL_TABLE_PATH}. "
                "Build it first with `python -m pitchseq.build_table` (RUNBOOK Step 1)."
            )
        table = pd.read_parquet(REAL_TABLE_PATH, engine="pyarrow")
        return table, {"world": "real"}

    from pitchseq.decision_table import build_decision_table
    from pitchseq.synth import make_null_world, make_positive_world

    if mode == "synth_null":
        raw, truth = make_null_world(n_games=200, seed=seed, innings_per_game=6)
    elif mode == "synth_positive":
        raw, truth = make_positive_world(
            n_games=300, seed=seed, innings_per_game=6,
            effect_size=0.30, velo_gap_threshold=5.0,
        )
    else:
        raise ValueError(f"unknown DATA_MODE {mode!r}")
    return build_decision_table(raw), truth


table, truth = load_decision_table(DATA_MODE, CONFIG, SEED)

splits = make_splits(table, CONFIG)["primary"]
train = table.loc[splits["train"].to_numpy()].reset_index(drop=True)
val = table.loc[splits["val"].to_numpy()].reset_index(drop=True)

print(f"world        : {truth.get('world', DATA_MODE)}")
print(f"total rows   : {len(table):,}")
print(f"train rows   : {len(train):,}   (seasons {CONFIG['split']['train']})")
print(f"val rows     : {len(val):,}   (seasons {CONFIG['split']['val']})")
print(f"columns      : {len(table.columns)}")
if truth.get("world") == "positive":
    print(f"planted lift : empirical_whiff_lift={truth.get('empirical_whiff_lift'):+.4f} "
          f"on affected_rate={truth.get('affected_rate'):.3f}")

**How to read the split.** On the synthetic worlds the season is assigned cyclically, so the
train/val split is populated even though the world is tiny. On real data `train` is ~2.3M
pitches and `val` ~0.77M. If either side is empty, the season column is wrong — stop and
check the build.

### (a) Family mix by count — a first look at selection structure

Before any model, the crudest form of "what does he throw?" is the family distribution inside
each ball–strike count. The heatmap below is `P(family | count)` — each column (a count) sums
to 1 across the 8 families. If sequencing mattered *at all* for selection, we would expect the
count alone to already move the mix (e.g. more sliders with two strikes). This is the **C**
view's worldview: context, no within-PA history.

In [ ]:
COUNTS = [(b, s) for b in range(4) for s in range(3)]  # 12 ball-strike states
count_labels = [f"{b}-{s}" for (b, s) in COUNTS]

mix = np.zeros((len(FAMILIES), len(COUNTS)))
fam_arr = train["family"].astype("object").to_numpy()
b_arr = pd.to_numeric(train["balls"], errors="coerce").to_numpy()
s_arr = pd.to_numeric(train["strikes"], errors="coerce").to_numpy()
for j, (b, s) in enumerate(COUNTS):
    sel = (b_arr == b) & (s_arr == s)
    if sel.sum() == 0:
        continue
    for i, f in enumerate(FAMILIES):
        mix[i, j] = np.mean(fam_arr[sel] == f)

fig, ax = new_fig(figsize=(8.2, 4.6))
im = ax.imshow(mix, aspect="auto", cmap="viridis", vmin=0.0, vmax=float(mix.max()))
ax.set_xticks(range(len(COUNTS)))
ax.set_xticklabels(count_labels, rotation=0)
ax.set_yticks(range(len(FAMILIES)))
ax.set_yticklabels(FAMILIES)
ax.set_xlabel("count (balls-strikes)")
ax.set_ylabel("pitch family")
ax.set_title("The pitch mix already shifts with the count\n(P(family | count), each column sums to 1)")
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
cbar.set_label("share of pitches")
plt.show()

*Caption.* Brighter cells = a larger share of that family in that count. Read down a column
to see the repertoire the count induces; read across a row to see how one family's usage rises
or falls with the count. Visible count structure here is exactly what the **C** view captures
without any sequencing — the bar every history view must clear to earn its complexity.

### (b) Plate-appearance length — where within-PA order can even exist

Order effects can only appear in plate appearances long enough to *have* an order. A PA that
ends on pitch 1 carries no within-PA history; the ordered **O** view can only differ from
**C** once `pitch_number ≥ 3` (it keys on the last *two* prior pitches). The histogram shows
how much of the data is even eligible.

In [ ]:
pa_len = train.groupby("pa_id")["pitch_number"].max().to_numpy()
fig, ax = new_fig()
bins = np.arange(1, int(pa_len.max()) + 2) - 0.5
ax.hist(pa_len, bins=bins, color=OKABE_ITO["sky_blue"], edgecolor="white")
ax.set_xlabel("pitches in the plate appearance")
ax.set_ylabel("number of plate appearances")
ax.set_title("Most order-sensitivity lives in longer plate appearances\n(O differs from C only once a PA reaches 3+ pitches)")
med = float(np.median(pa_len))
ax.axvline(med, color=OKABE_ITO["vermillion"], lw=2)
ax.text(med + 0.1, ax.get_ylim()[1] * 0.9, f"median = {med:.0f}", color=OKABE_ITO["vermillion"])
plt.show()

*Caption.* The mass to the left of pitch 3 is PAs where **O** cannot differ from **C** — the
ordered view has nothing to condition on. The longer the tail, the more room ordered history
has to matter. Keep this in mind when reading the `long_pa` (`pitch_number ≥ 3`) slice later.

### (c) Reward by outcome — the sign check, made visual

The reward is `R = -delta_run_exp`, defined so **larger is better for the pitcher** (SPEC §5).
Before trusting any run-value table we confirm the sign the way SPEC §5 mandates: `R` should
be **positive** on called strikes / whiffs / outs and **negative** on balls / walks / home
runs. The boxplot makes the sign check something you can see at a glance.

In [ ]:
R = pd.to_numeric(train["R"], errors="coerce").to_numpy()
o1 = train["outcome1"].astype("object").to_numpy()
groups = [c for c in OUTCOME1 if np.isfinite(R[o1 == c]).any()]
data = [R[(o1 == c) & np.isfinite(R)] for c in groups]

fig, ax = new_fig()
bp = ax.boxplot(data, vert=True, showfliers=False, patch_artist=True,
                medianprops=dict(color=OKABE_ITO["black"]))
for patch in bp["boxes"]:
    patch.set_facecolor(OKABE_ITO["sky_blue"])
    patch.set_alpha(0.7)
ax.axhline(0.0, color=OKABE_ITO["vermillion"], lw=1.5, ls="--")
ax.set_xticklabels(groups, rotation=20, ha="right")
ax.set_ylabel("reward  R = -delta_run_exp  (runs)")
ax.set_title("Reward sign is correct: strikes/whiffs help the pitcher, balls hurt\n(dashed line = 0; boxes should sit above 0 for strikes, below for balls)")
plt.show()

*Caption.* Boxes above the dashed zero line are pitcher-positive outcomes (called strikes,
whiffs, in-play outs dominate that bucket); boxes below are pitcher-negative (balls). If any
sign were flipped, every downstream run-value number would be inverted — this is the cheap
visual guard.

### (d) Repertoire diversity — how much a pitcher table can even personalise

WS1's tables include a **pitcher** level: within a count cell, they shrink each pitcher toward
the count-and-handedness average. That level is only useful if pitchers actually differ. The
histogram of per-pitcher family **entropy** (in bits) shows the spread of repertoire breadth —
a one-pitch specialist has low entropy, a five-pitch mixer has high entropy.

In [ ]:
def family_entropy(fam_labels):
    vals, counts = np.unique(fam_labels, return_counts=True)
    p = counts / counts.sum()
    return float(-(p * np.log2(p)).sum())

ent = train.groupby("pitcher")["family"].apply(lambda s: family_entropy(s.astype("object").to_numpy()))
ent = ent.to_numpy()

fig, ax = new_fig()
ax.hist(ent, bins=20, color=OKABE_ITO["bluish_green"], edgecolor="white")
ax.set_xlabel("repertoire entropy per pitcher (bits over 8 families)")
ax.set_ylabel("number of pitchers")
ax.set_title("Pitchers differ in repertoire breadth, so the pitcher level carries signal\n(higher entropy = a more balanced multi-pitch mix)")
plt.show()

*Caption.* Spread here is what justifies the pitcher level of the hierarchy: if every pitcher
had the same mix, shrinking toward the count average would lose nothing. On real data expect a
broad spread (specialists to kitchen-sink arsenals); on the synthetic worlds the six pitcher
archetypes produce a handful of clusters.

### (e) The cell-count explosion — a preview of the support problem

Here is the single most important picture in WS1, previewed. As the conditioning key deepens
from **C** (count × handedness × pitcher) to the history views, the number of **distinct
cells** the table must estimate multiplies. The bar chart (log scale, view colours) counts the
distinct *deepest* cells each view creates on the training data. Section 7 turns this into the
full support exhibit; here it is just the shape of the problem.

In [ ]:
def deepest_cell_count(t, view):
    '''Distinct deepest selection cells for a view: (ch, pit) for C, (ch, pit, hist) else.

    Mirrors the selection key ladder in workstreams.ws1_eb_tables.model exactly, using the
    public count_hand_code / history_signature helpers.
    '''
    ch = ebm.count_hand_code(t)
    pit = pd.to_numeric(t["pitcher"], errors="coerce").fillna(-1).astype("int64").to_numpy()
    if view == "C":
        keys = list(zip(ch.tolist(), pit.tolist()))
    else:
        hist = ebm.history_signature(t, view)
        keys = list(zip(ch.tolist(), pit.tolist(), hist.tolist()))
    return len(set(keys))

cell_counts = {v: deepest_cell_count(train, v) for v in VIEWS}

fig, ax = new_fig()
xs = range(len(VIEWS))
bars = ax.bar(xs, [cell_counts[v] for v in VIEWS],
              color=[VIEW_COLORS[v] for v in VIEWS])
ax.set_yscale("log")
ax.set_xticks(list(xs))
ax.set_xticklabels(VIEWS)
ax.set_xlabel("state view")
ax.set_ylabel("distinct deepest cells (log scale)")
ax.set_title("Deepening the key multiplies the cells to estimate\n(the support problem, previewed — note the log axis)")
for x, v in zip(xs, VIEWS):
    ax.text(x, cell_counts[v] * 1.1, f"{cell_counts[v]:,}", ha="center", va="bottom", fontsize=10)
plt.show()

*Caption.* A log axis, and still the history views tower over **C**. Each extra key coordinate
splits every existing cell into many. The counts are finite because unseen keys **back off**
to their parent rather than being estimated from nothing — but the more a view relies on
backoff, the less its "extra" resolution is doing. That trade-off is the whole story of §7.

## 3. The two models (exposition)

No code in this section — just the two formulas WS1 rests on, in words and symbols. Full
step-by-step derivations are in `THEORY.md`; the exact code is in
`workstreams/ws1_eb_tables/model.py`.

### The classic intuition: batting-average shrinkage

The oldest and cleanest version of this idea is Efron and Morris's baseball example. A hitter
who goes 9-for-30 early in the season has a raw average of .300, but you should not believe
.300 — 30 at-bats is thin evidence. The **shrinkage** estimate pulls that .300 toward the
league mean (say .260), and by how much depends on how little data you have: 30 at-bats gets
pulled hard; 300 at-bats barely moves. The pulled-together estimates famously predict the rest
of the season better than the raw averages do. WS1 is that idea applied twice — once to a
*distribution* over pitch families (selection), once to a *mean* reward (run value) — and
stacked into a hierarchy so a thin cell borrows from its parent instead of the global mean.

### Selection: Dirichlet-multinomial posterior mean

For a cell `c` (a specific count × handedness × pitcher × history key) with observed family
counts `n_c = (n_{c,1}, …, n_{c,8})`, `N_c = Σ_k n_{c,k}`, and a parent distribution
`π_parent(c)` over the 8 families, the posterior-mean family distribution is

$$\hat p_c = \frac{n_c + \alpha_l\,\pi_{\text{parent}(c)}}{N_c + \alpha_l}.$$

`α_l` is the **concentration** of level `l` — a prior pseudo-count strength, the "how many
imaginary pitches drawn from the parent do we add?" knob. Rearranged (see §3 of `THEORY.md`),
this is a convex combination

$$\hat p_c = \underbrace{\frac{N_c}{N_c+\alpha_l}}_{\text{weight on the data}}\,\hat p^{\text{MLE}}_c
          + \underbrace{\frac{\alpha_l}{N_c+\alpha_l}}_{\text{weight on the parent}}\,\pi_{\text{parent}(c)},$$

exactly the batting-average story: a well-populated cell (`N_c ≫ α_l`) trusts its own tallies;
a thin cell leans on the parent. An **unseen** cell (`N_c = 0`) returns the parent exactly —
that *is* backoff.

### Run value: hierarchical normal partial pooling

For the same cell with `n_c` observations, sample-mean reward `R̄_c`, and parent posterior mean
`m_parent(c)`, the posterior-mean reward is

$$\hat m_c = \frac{n_c}{n_c+\kappa_l}\,\bar R_c + \frac{\kappa_l}{n_c+\kappa_l}\,m_{\text{parent}(c)},$$

with the analogous shrinkage strength `κ_l` (a prior sample size). Its posterior standard
deviation, with a pooled within-cell reward variance `σ²`, is

$$\text{sd}(\hat m_c) = \frac{\sigma}{\sqrt{n_c+\kappa_l}},$$

so uncertainty falls like `1/√(n_c+κ_l)` — thin cells come with honestly wide error bars. WS1
reports that uncertainty, which is exactly what the later bandit (WS4) will want.

### What a *fitted* concentration means

We do not guess `α_l` / `κ_l`; we **fit** each level's value by empirical Bayes (maximum
marginal likelihood, with a method-of-moments fallback), reading it straight off how much the
cells at that level actually disagree:

- **`α_l → ∞` (or `κ_l → ∞`)** means *"this level adds nothing"* — the child cells look like
  independent draws from the parent, so the fit pools them all the way back. You will see this
  literally happen on the null world, where the fitted history-level `α` runs to its ceiling.
- **`α_l → 0`** means *"the cells are genuinely heterogeneous"* — trust each cell's own data,
  shrink little.

The fitted concentration is therefore not a nuisance parameter but a **readout**: it tells you,
per level, whether that slice of conditioning carries signal or noise.

## 4. Selection tables (next-pitch family)

Now we fit. For each view we build the hierarchical Dirichlet-multinomial selection table on
`train`, predict the next-pitch family distribution on `val`, and keep both the fitted model
(for its diagnostics) and a standard-schema prediction table (for the shared harness). We fit
each view **once** here and reuse the fitted models and predictions throughout the rest of the
notebook.

In [ ]:
def selection_prediction_df(model, val_table, view):
    proba = model.predict_proba(val_table)
    df = pd.DataFrame({"row_id": val_table["row_id"].to_numpy()})
    for j, col in enumerate(ACTION_PROB_COLS):
        df[col] = proba[:, j]
    df["model_id"] = "ws1_eb_tables"
    df["state_view"] = view
    df["seconds"] = 0.0
    df["peak_mem_mb"] = 0.0
    df["n_params"] = int(model.n_params)
    return df

sel_models = {}
sel_pred = {}
for v in VIEWS:
    m = ebm.fit(train, v, target="selection")
    sel_models[v] = m
    sel_pred[v] = selection_prediction_df(m, val, v)

print("fitted selection tables for views:", list(sel_models))

### Fitted concentrations, level by level

This is the readout promised in §3. For each view we print the fitted concentration `α` at
every level of the hierarchy and the estimator that produced it (`mml` = maximum marginal
likelihood, `mom` = method-of-moments fallback, `default` = too few cells to identify). Watch
the **history** level: a value pinned near the ceiling (`≈ 1e6`) is the model telling you the
within-PA history adds essentially nothing to *selection* beyond the pitcher × count cell.

In [ ]:
rows = []
for v in VIEWS:
    m = sel_models[v]
    conc = m.concentrations_
    meth = m.concentration_methods_
    for level in conc:
        rows.append({
            "view": v, "level": level,
            "alpha": conc[level], "method": meth[level],
            "n_cells": m.n_cells_[level],
        })
conc_table = pd.DataFrame(rows)
with pd.option_context("display.float_format", lambda x: f"{x:,.3g}"):
    print(conc_table.to_string(index=False))

**How to read it.** The `global` and (for a single-cell root) some levels use the weak default
prior — ignore those. The informative rows are `count_hand`, `pitcher`, and `history`. A large
fitted `pitcher` concentration means pitchers differ little within a count after shrinkage; a
large `history` concentration means order adds little to *what is thrown next*. On the null
world the history `α` sits at its `1e6` ceiling **by design of the data**, not by a bug: the
simulator's only order habit (a mild no-three-in-a-row tendency) is too weak for a flat table
to resolve, so empirical Bayes correctly pools it away.

### Selection log loss versus the count-based references

We score every view through the **shared harness** — no bespoke scoring — and line them up
against the four count-based reference baselines (`eval/baselines.py`): global family
frequency by count × hand, pitcher × count, first-order transition, and pitcher × count ×
prev-family. WS1 is the *serious* hierarchical version of the same idea, so it should not lose
to these quick references. Log loss is the primary selection metric (lower is better).

In [ ]:
sel_reports = {
    v: evaluate_predictions(sel_pred[v], val, config=CONFIG, n_boot=N_BOOT, seed=SEED)
    for v in VIEWS
}
sel_loss = {v: sel_reports[v]["slices"]["all"]["action_prob"]["log_loss"] for v in VIEWS}

baseline_loss = {}
for name, mdl in build_baselines(alpha=8.0).items():
    mdl.fit(train)
    proba = mdl.predict_proba(val)
    bdf = pd.DataFrame({"row_id": val["row_id"].to_numpy()})
    for j, col in enumerate(ACTION_PROB_COLS):
        bdf[col] = proba[:, j]
    bdf["model_id"] = f"baseline_{name}"
    bdf["state_view"] = "NA"
    bdf["seconds"] = 0.0
    bdf["peak_mem_mb"] = 0.0
    bdf["n_params"] = 0
    brep = evaluate_predictions(bdf, val, config=CONFIG, n_boot=N_BOOT, seed=SEED)
    baseline_loss[name] = brep["slices"]["all"]["action_prob"]["log_loss"]

print("WS1 selection log loss (val):")
for v in VIEWS:
    print(f"  {v:<3}: {sel_loss[v]:.4f}")
print("count-based references:")
for k, x in baseline_loss.items():
    print(f"  {k:<20}: {x:.4f}")

In [ ]:
fig, ax = new_fig(figsize=(7.6, 4.4))
xs = np.arange(len(VIEWS))
ax.bar(xs, [sel_loss[v] for v in VIEWS], color=[VIEW_COLORS[v] for v in VIEWS], width=0.6)
for x, v in zip(xs, VIEWS):
    ax.text(x, sel_loss[v], f"{sel_loss[v]:.3f}", ha="center", va="bottom", fontsize=10)

# Reference baselines as horizontal lines, directly labelled at the right edge.
ref_styles = {
    "global_count_hand": (":", "global x count x hand"),
    "pitcher_count": ("--", "pitcher x count"),
    "transition": ("-.", "prev-family x count"),
    "pitcher_count_prev": ("-", "pitcher x count x prev"),
}
for name, (ls, label) in ref_styles.items():
    if name in baseline_loss:
        y = baseline_loss[name]
        ax.axhline(y, color=REF_COLOR, ls=ls, lw=1.2, alpha=0.8)
        ax.text(len(VIEWS) - 0.4, y, f"  {label}", va="center", ha="left", fontsize=8, color=REF_COLOR)

ax.set_xticks(xs)
ax.set_xticklabels(VIEWS)
ax.set_xlabel("state view")
ax.set_ylabel("selection log loss (val, lower better)")
ax.set_title("WS1 tables should sit at or below the quick count references\n(bars = WS1 views; black lines = count-based baselines)")
ax.set_xlim(-0.6, len(VIEWS) + 0.9)
plt.show()

*Caption.* Bars are WS1's views; black lines are the reference baselines. The history views
(U/L1/O) should sit **at or below** the `pitcher × count × prev` line — if the serious
shrinkage lost to the quick baseline, something is wrong. Bars that barely differ from each
other say history is not sharpening *selection* much beyond count and pitcher. On the synthetic
worlds `stand / p_throws` carry no signal, so **C** can sit a hair above the `pitcher × count`
line (a documented, world-specific artifact of ~0.009 log loss that is expected to reverse on
real data, where handedness matters).

### Confidence reliability for the ordered (O) view

A good probabilistic model is **calibrated**: when it says "70% four-seam", four-seam should
follow about 70% of the time. The reliability curve bins the model's top-class confidence and
plots, per bin, the mean predicted confidence against the observed hit rate. Points on the
diagonal are perfectly calibrated; below the diagonal is over-confidence.

In [ ]:
proba_O = sel_models["O"].predict_proba(val)
conf = proba_O.max(axis=1)
pred_fam = np.array(FAMILIES)[proba_O.argmax(axis=1)]
correct = (pred_fam == val["family"].astype("object").to_numpy()).astype(float)
rel = reliability_table(correct, conf, n_bins=10)

fig, ax = new_fig(figsize=(5.4, 5.2))
ax.plot([0, 1], [0, 1], ls="--", color=OKABE_ITO["black"], lw=1, label="perfect calibration")
ax.plot(rel["mean_pred"], rel["frac_pos"], "-o", color=VIEW_COLORS["O"], lw=2)
for _, r in rel.iterrows():
    ax.annotate(f"n={int(r['count'])}", (r["mean_pred"], r["frac_pos"]),
                textcoords="offset points", xytext=(4, -9), fontsize=7, color="gray")
ax.set_xlabel("mean predicted confidence (top class)")
ax.set_ylabel("observed hit rate")
ax.set_title("Is the O view calibrated?\n(points on the dashed line = confidence matches reality)")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.show()

*Caption.* Bin counts are annotated so you can discount sparsely-populated bins. Systematic
sag below the diagonal at high confidence is over-confidence (a table certain about a cell it
has barely seen) — cross-reference with the support exhibit in §7.

### Per-slice selection loss

SPEC §6 requires every sequence metric reported separately across slices: all pitches,
sequence-eligible (`t≥2`), long PAs (`t≥3`), two-strike, three-ball, first-pitch. The table
below is each view's selection log loss per slice. The `first_pitch` slice is a useful sanity
check: with no prior pitch, U/L1/O *must* equal C there.

In [ ]:
slice_names = list(CONFIG.get("report_slices", ["all"]))
per_slice_loss = {}
for v in VIEWS:
    sl = sel_reports[v]["slices"]
    per_slice_loss[v] = {
        s: (sl[s]["action_prob"]["log_loss"] if (s in sl and "action_prob" in sl[s]) else np.nan)
        for s in slice_names
    }
slice_df = pd.DataFrame(per_slice_loss).T[slice_names]
slice_df.index.name = "view"
print(slice_df.to_string(float_format=lambda x: f"{x:.4f}"))

**How to read it.** Compare a view's loss across slices to see where any sequencing edge
concentrates — if ordered history helps at all, it should help most in `long_pa` and
`two_strike`, not in `first_pitch`. Identical columns across views mean the views are making
identical predictions on that slice (expected on `first_pitch`, and — as we will see — expected
*everywhere* on the null world, where the tables collapse the history level away).

## 5. Run-value tables (E[R | cell, pitch thrown])

The second table family estimates the expected reward of a pitch, **conditioned on the pitch
actually thrown** (decision D22: outcome models condition on the current action, keeping
finding #1 out of finding #2). So every run-value cell carries the current `family` as an extra
key coordinate. We fit one hierarchical-normal table per view, predict `(E[R], sd)` on `val`,
and score reward calibration through the harness.

In [ ]:
def run_value_prediction_df(model, val_table, view):
    mean, sd = model.predict_run_value(val_table)
    df = pd.DataFrame({"row_id": val_table["row_id"].to_numpy()})
    df["exp_reward"] = mean
    df["exp_reward_sd"] = sd
    df["model_id"] = "ws1_eb_tables"
    df["state_view"] = view
    df["seconds"] = 0.0
    df["peak_mem_mb"] = 0.0
    df["n_params"] = int(model.n_params)
    return df

rv_models = {}
rv_pred = {}
rv_reports = {}
for v in VIEWS:
    m = ebm.fit(train, v, target="run_value")
    rv_models[v] = m
    rv_pred[v] = run_value_prediction_df(m, val, v)
    rv_reports[v] = evaluate_predictions(rv_pred[v], val, config=CONFIG, n_boot=N_BOOT, seed=SEED)

print("run-value tables (val): MAE / RMSE / calibration slope & intercept")
for v in VIEWS:
    blk = rv_reports[v]["slices"]["all"]["exp_reward"]
    cal = blk["calibration"]
    print(f"  {v:<3}: MAE={blk['mae']:.4f}  RMSE={blk['rmse']:.4f}  "
          f"slope={cal['slope']:.3f}  intercept={cal['intercept']:+.4f}  R2={cal['r2']:.3f}")

**How to read it.** Reward here is a change in run expectancy — a small number in runs, and an
extremely noisy one at the single-pitch level, so the absolute MAE looks large relative to the
signal; that is the nature of one-pitch run values, not a defect. The **calibration** line is
the honest read: an OLS fit of observed reward on predicted reward should have **slope ≈ 1** and
**intercept ≈ 0**. A slope well below 1 signals over-confident spread; the tables' shrinkage is
designed to keep the slope sane by not chasing noise in thin cells.

### The shrinkage picture: posterior sd falls like 1/√(n+κ)

This scatter *is* the shrinkage story of §3, drawn from the fitted O run-value table. Each
point is a validation row: on the x-axis, the training support `N` of the cell it resolved to;
on the y-axis, the posterior standard deviation the table reports. Because
`sd = σ/√(N+κ)`, the cloud must hug a `1/√(N+κ)` curve — thin cells (small `N`) get honestly
large error bars, well-populated cells get tight ones.

In [ ]:
v = "O"
m = rv_models[v]
mean_v, sd_v = m.predict_run_value(val)
N_v = m.resolved_train_n(val)
kappa_hist = m.concentrations_.get("history", np.nan)
sigma = float(np.sqrt(m.sigma2_)) if np.isfinite(m.sigma2_) else np.nan

# Subsample for a legible scatter if val is large.
rng = np.random.default_rng(SEED)
idx = np.arange(len(val))
if len(idx) > 4000:
    idx = rng.choice(idx, size=4000, replace=False)

fig, ax = new_fig()
ax.scatter(N_v[idx], sd_v[idx], s=8, alpha=0.25, color=VIEW_COLORS["O"], edgecolors="none")
if np.isfinite(sigma) and np.isfinite(kappa_hist):
    ng = np.linspace(max(1, np.nanmin(N_v)), np.nanpercentile(N_v, 99), 200)
    ax.plot(ng, sigma / np.sqrt(ng + kappa_hist), color=OKABE_ITO["black"], lw=2,
            label=fr"$\sigma/\sqrt{{N+\kappa}}$, $\kappa$={kappa_hist:,.0f}")
    ax.legend(frameon=False, fontsize=9)
ax.set_xscale("log")
ax.set_xlabel("training support N of the resolved cell (log scale)")
ax.set_ylabel("posterior sd of E[R]  (runs)")
ax.set_title("More data, tighter estimate: posterior sd falls like 1/sqrt(N + kappa)\n(each point = one validation pitch, O run-value table)")
plt.show()

*Caption.* The downward sweep is shrinkage doing its job: the table never claims more certainty
than its support warrants. Points stacked at the left edge (tiny `N`) are rows that resolved to
thinly-populated deep cells — the same rows the support exhibit (§7) flags. The solid curve is
the theoretical `σ/√(N+κ)` using the fitted history-level `κ`; scatter around it comes from rows
that backed off to shallower levels with different support.

## 6. The ablation — Δ_order with a clustered confidence interval

The headline sequencing comparison (SPEC §6) is

$$\Delta_{\text{order}} = \text{Loss}(\min[\,U, L1\,]) - \text{Loss}(O).$$

In words: take the *better* of the two "history-lite" views (unordered counts, or just the
previous pitch), and ask how much the **fully ordered** view improves on it. WS1 computes this
on the **selection** target (next-pitch log loss) through the shared `compare_views`, with a
confidence interval bootstrapped over **pitcher-game clusters** (SPEC §7) — because pitches in
the same game are dependent, a naive row-level CI would be far too narrow.

In [ ]:
cv = compare_views(sel_pred, val, config=CONFIG, target="family", n_boot=N_BOOT, seed=SEED)
delta_order = cv["deltas"]["delta_order"]
do_ci = cv["delta_order_ci"]
print("per-view selection loss (aligned to common rows):")
for v, x in cv["losses"].items():
    print(f"  {v:<3}: {x:.5f}")
print(f"\nDelta_order = min(Loss_U, Loss_L1) - Loss_O = {delta_order:+.5f}")
print(f"95% clustered CI: [{do_ci['lo']:+.5f}, {do_ci['hi']:+.5f}]   (n_boot={do_ci['n_boot']})")
print(f"significantly positive (CI lower bound > 0)? {do_ci['lo'] > 0}")

### The D21 reading rule (binding on this workstream)

`Δ_order` uses `min[U, L1]`, and the minimum of two noisy losses is **optimistically biased
low** — so `Δ_order` is **negatively biased under the null**. This has one binding consequence
(decision **D21**), which every interpretation in §9 obeys:

- The null criterion is **"not significantly positive"**, i.e. the CI lower bound is **not**
  above 0. It is *not* "equal to zero".
- A small **negative** `Δ_order` on real data reads as **consistent with no ordering effect** —
  never as "order *hurts*". Order cannot hurt an out-of-sample predictor that is free to ignore
  it; a negative value is the optimism bias, not a real cost.
- Only a **significantly positive** `Δ_order` (CI lower bound > 0) claims genuine ordered
  structure.

A WS1-specific sharpening: because WS1's flat shrinkage pools the history level away when it is
uninformative, on the null world every view makes the *same* prediction and `Δ_order` is
**exactly 0** — there is no optimism bias to fight, because there is no history resolution at
all. That makes a *clearly negative* WS1 `Δ_order` meaningful in a way it would not be for a
flexible model: for tables, a real negative signals **over-fragmentation** (§9, branch R3).

### Per-slice Δ_order

We recompute the ablation within each reporting slice, to see whether any ordered edge
concentrates where it should (longer PAs, two-strike counts) rather than smearing uniformly.

In [ ]:
masks = slice_masks(val, slice_names)
per_slice_delta = {}
for sname, mask in masks.items():
    if mask.sum() < 200:
        per_slice_delta[sname] = np.nan
        continue
    sub_val = val.loc[mask].reset_index(drop=True)
    keep_ids = set(sub_val["row_id"].tolist())
    sub_pred = {v: sel_pred[v][sel_pred[v]["row_id"].isin(keep_ids)] for v in ("U", "L1", "O")}
    try:
        cvs = compare_views(sub_pred, sub_val, config=CONFIG, target="family",
                            n_boot=max(20, N_BOOT // 2), seed=SEED)
        per_slice_delta[sname] = cvs["deltas"]["delta_order"]
    except (KeyError, ValueError):
        per_slice_delta[sname] = np.nan

for s in slice_names:
    val_d = per_slice_delta.get(s, np.nan)
    shown = f"{val_d:+.5f}" if np.isfinite(val_d) else "   n/a"
    print(f"  {s:<12}: Delta_order = {shown}")

**How to read it.** On the `first_pitch` slice `Δ_order` is 0 by construction (no history to
order). Elsewhere, an ordered effect that is real should be *largest* in `long_pa` and
`two_strike`. A `Δ_order` that is uniformly ~0 across every slice — the expected WS1 result — is
the flat table saying it cannot find ordered *selection* structure, which is precisely why the
project ships WS2 and WS3 to look harder.

## 7. The support-problem exhibit — WS1's headline contribution

This is what WS1 exists to make undeniable. The `support_table` helper profiles, per view, how
badly the conditioning key fragments the data and how much of the evaluation set lands in cells
too thin to estimate by counting. We compute it once and read it three ways.

In [ ]:
support = ebm.support_table(train, val, views=VIEWS, target="selection")
cols_show = ["view", "n_cells", "cell_n_p50", "cell_n_mean",
             "frac_eval_lt5", "frac_eval_lt20", "frac_eval_lt50"]
print(support[cols_show].to_string(index=False, float_format=lambda x: f"{x:.3f}"))

### (a) How many cells each view creates

In [ ]:
fig, ax = new_fig()
xs = np.arange(len(support))
ax.bar(xs, support["n_cells"].to_numpy(),
       color=[VIEW_COLORS[v] for v in support["view"]])
ax.set_yscale("log")
ax.set_xticks(xs)
ax.set_xticklabels(support["view"].tolist())
ax.set_xlabel("state view")
ax.set_ylabel("distinct deepest cells (log scale)")
ax.set_title("Every history coordinate multiplies the cells to estimate\n(log scale — the history views dwarf C)")
for x, n in zip(xs, support["n_cells"].to_numpy()):
    ax.text(x, n * 1.1, f"{int(n):,}", ha="center", va="bottom", fontsize=9)
plt.show()

*Caption.* The same explosion previewed in §2(e), now from the model's own cell inventory. Note
that the ordering of **U** versus **O** is *world-dependent*: **U** keys the full (capped)
multiset of priors while **O** keys only the ordered last two, so on longer plate appearances
**U** can fragment into more cells than **O**. The robust signal is not their order but that
both tower over **C**.

### (b) How much of the evaluation set lands in thin cells

In [ ]:
fig, ax = new_fig(figsize=(7.8, 4.4))
thresholds = [("frac_eval_lt5", "n < 5"), ("frac_eval_lt20", "n < 20"), ("frac_eval_lt50", "n < 50")]
xs = np.arange(len(support))
width = 0.26
shades = [OKABE_ITO["vermillion"], OKABE_ITO["orange"], OKABE_ITO["yellow"]]
for k, (col, label) in enumerate(thresholds):
    ax.bar(xs + (k - 1) * width, support[col].to_numpy(), width=width,
           color=shades[k], label=label, edgecolor="white")
ax.set_xticks(xs)
ax.set_xticklabels(support["view"].tolist())
ax.set_xlabel("state view")
ax.set_ylabel("fraction of eval rows in low-support cells")
ax.set_title("Deeper keys push more of the eval set into thin cells\n(each group: fraction of validation pitches whose cell had < 5 / 20 / 50 training pitches)")
ax.legend(frameon=False, title="cell training support")
plt.show()

*Caption.* This is the fragmentation cost in the currency that matters — *evaluation exposure*.
The `n < 20` fraction climbing from a few percent at **C** into the 0.3–0.5 range for the
history views is the concrete statement of why deeper keys cannot simply be counted: a third to
a half of the validation pitches would be predicted from a handful of training examples, if the
table did not back off.

### (c) How often each view falls back on backoff

In [ ]:
backoff_cols = [c for c in support.columns if c.startswith("backoff_")]
fig, ax = new_fig(figsize=(7.8, 4.4))
xs = np.arange(len(support))
bottom = np.zeros(len(support))
level_order = ["backoff_global", "backoff_count_hand", "backoff_pitcher", "backoff_history"]
level_order = [c for c in level_order if c in backoff_cols] + [c for c in backoff_cols if c not in level_order]
level_shades = [OKABE_ITO["yellow"], OKABE_ITO["sky_blue"], OKABE_ITO["bluish_green"], OKABE_ITO["vermillion"]]
for k, col in enumerate(level_order):
    vals = support[col].fillna(0.0).to_numpy()
    ax.bar(xs, vals, bottom=bottom, label=col.replace("backoff_", ""),
           color=level_shades[k % len(level_shades)], edgecolor="white")
    bottom += vals
ax.set_xticks(xs)
ax.set_xticklabels(support["view"].tolist())
ax.set_xlabel("state view")
ax.set_ylabel("fraction of eval rows resolved at each level")
ax.set_title("Where predictions actually resolve: the deeper the key, the more it leans on backoff\n(a short deepest-level bar = the extra resolution is rarely used)")
ax.legend(frameon=False, title="resolved at level", fontsize=8, ncol=2)
plt.show()

*Caption.* Each bar splits the validation set by the level that actually supplied the
prediction. A **history** view whose `history` segment is short is telling you that most of its
"extra" cells were never seen in training, so predictions fell back to the pitcher (or count)
level — the deep resolution is mostly decorative. That is the mechanism behind a flat Δ_order.

### Why OM is table-infeasible — the argument in numbers

The matchup-memory view **OM** would key on the specific batter-versus-pitcher history. `support_note`
computes the fragmentation that makes this hopeless for a pure table: an average pitcher–batter
pair has only a handful of career pitches (SPEC §3.4 says ~20), so almost every OM cell would
have near-zero support and back off to O, adding nothing. This is why WS1 declares
`Δ_matchup = N/A` and hands matchup memory to the pooled/embedding models (WS3+).

In [ ]:
note = ebm.support_note(train)
print(note["message"])
print()
print(f"pitcher-batter pairs           : {note.get('n_pairs', 'n/a'):,}" if "n_pairs" in note else "")
if "pair_pitches_median" in note:
    print(f"median pitches per pair        : {note['pair_pitches_median']:.1f}")
    print(f"mean pitches per pair          : {note['pair_pitches_mean']:.1f}")
    print(f"fraction of pairs with < 20    : {note['frac_pairs_lt20']:.3f}")
    print(f"fraction of pairs with < 50    : {note['frac_pairs_lt50']:.3f}")
    print(f"pitcher-batter-count cells     : {note['n_pair_count_cells']:,}")
    print(f"  fraction of those with < 5   : {note['frac_pair_count_cells_lt5']:.3f}")

**How to read it.** If the fraction of pitcher–batter pairs with fewer than 20 pitches is high
(it will be — the vast majority), then an OM table is estimating almost every cell from nothing.
Declaring OM infeasible is not a shortcut; it is the honest statement of a hard data limit, and
it is one of WS1's deliverables to the project.

## 8. Falsification — the two synthetic signatures

A model that merely *runs* has proven nothing (SPEC §0). The correctness oracle gives us two
worlds with known truth, and WS1 must produce the right signature on each. These cells run the
**whole WS1 pipeline** on the null and positive worlds (fast, no data files) and print the two
signatures. They run regardless of `DATA_MODE`, because they are the workstream's self-test.

### Null world — no ordered outcome effect

The null world has order-dependent *selection* habits but its outcomes depend only on
`(count, platoon, current family)` — **no** ordered outcome effect exists to find. The correct
WS1 signature: the fitted history-level `α` runs to its ceiling, every view collapses to the
same prediction, and the selection `Δ_order` is **exactly 0**.

In [ ]:
null_res = run_ws1(synth="null", n_games=90, seed=7, target="selection",
                   n_boot=50, write_outputs=False)
nd = null_res["delta_order"]
print("NULL WORLD")
print(f"  Delta_order (selection) = {nd['point']:+.6f}   CI[{nd['ci']['lo']:+.5f}, {nd['ci']['hi']:+.5f}]")
print(f"  significantly positive? {nd['significantly_positive']}")
print(f"  reading: {nd['interpretation']}")
print("  fitted history-level alpha per history view (selection):")
for v in ("U", "L1", "O"):
    fa = null_res["fitted"][v]["selection"]["alpha"]
    fm = null_res["fitted"][v]["selection"]["alpha_method"]
    print(f"    {v:<3}: history alpha = {fa.get('history', float('nan')):,.0f}  ({fm.get('history')})")

**What this proves.** The history `α` pinned at ~`1e6` is empirical Bayes reporting *"the
within-PA history level adds nothing to selection"* — it pools every history cell back to its
pitcher × count parent, so U, L1 and O make identical predictions and `Δ_order = 0` exactly.
This is the model correctly finding **no ordered structure that is not there**. It also
demonstrates the WS1-specific point from §6: because the tables pool the history level away
under the null, there is no `min[U,L1]` optimism bias to correct — the null Δ_order is a clean
zero, not a small negative.

### Positive world — a planted ordered transition effect

The positive world adds one thing: when the velocity *change into the previous pitch*,
`|velo_{t-1} - velo_{t-2}|`, is large, the whiff probability on the current pitch is boosted by
a known amount. The effect is keyed on the **ordered transition** (not a single previous pitch)
precisely so that only the ordered **O** view can represent it — a pure previous-pitch effect
would already be captured by L1 and could never separate O from L1 (decision D21). We check that
the run-value tables recover the effect's **direction** with the heavily **attenuated**
magnitude decision D27 predicts.

In [ ]:
pos_res = run_ws1(synth="positive", n_games=300, seed=7, target="run_value",
                  n_boot=50, write_outputs=False)
rec = pos_res["positive_recovery"]
print("POSITIVE WORLD")
print(f"  true empirical reward lift on triggered rows : {rec['true_lift']:+.5f}")
print(f"  triggered rows / baseline rows               : {rec['n_triggered']} / {rec['n_base']}")
print("  per-view predicted E[R] edge (triggered - baseline), and the part beyond C:")
for v in ("C", "L1", "U", "O"):
    pv = rec["per_view"][v]
    print(f"    {v:<3}: edge={pv['edge']:+.6f}   beyond-C={pv['edge_vs_C']:+.6f}   "
          f"attenuation(edge/true_lift)={pv['attenuation']:.3f}")

**What this proves.** The recovery signature is **O > U > L1 ≈ C ≈ 0** in the *beyond-C* column,
with O's raw edge only ~**3%** (`attenuation ≈ 0.03`) of the planted lift. Both halves matter:

- **Direction is right, and only the ordered view sees it.** O recovers more of the effect than
  U, and far more than L1 (which, keying on a single previous pitch, cannot represent a
  transition *into* that pitch). This is finding #2 appearing exactly where the ground truth put
  it.
- **The magnitude is tiny, and that is the lesson — mechanism-blindness.** The planted effect is
  a *velocity* mechanism, but a family table can only see velocity **through the family →
  velocity-band correlation** (a curve is slow, a four-seam is fast). It has no `velo` column;
  it proxies the trigger through the ordered *family pair*, which is a lossy stand-in. So even a
  correctly-specified table recovers only a sliver of a real effect. Measured here at ~3%. When
  WS1's real-data `Δ_order` comes back near zero (§9), *this* is why "near zero" is **not** proof
  of absence — a table keyed on pitch names is partially blind to a mechanism keyed on pitch
  physics. WS3's feature model and WS6's learned representation exist to close that gap.

## 9. Results — branched interpretation

This is the section that turns numbers into a verdict. The code cell below inspects the
quantities computed above and prints **which branch applies**; the markdown that follows holds
the pre-written interpretation for every branch, so the conclusion is ready the instant the
cells finish. Two axes:

- **Selection** (finding #1): do the history views sharpen next-pitch prediction beyond
  context? Branches **S1 / S2**.
- **Δ_order** (the ordered-refinement test, measured on selection loss with a CI): **R1 / R2 /
  R3**, read under the D21 rule.

In [ ]:
# --- selection axis: do history views beat C? ---
C_loss = sel_loss["C"]
hist_losses = {v: sel_loss[v] for v in ("U", "L1", "O")}
best_hist_view = min(hist_losses, key=hist_losses.get)
best_hist = hist_losses[best_hist_view]
selection_gain = C_loss - best_hist  # positive => history helps selection

# A material improvement threshold (log loss units). Documented, not tuned: ~0.003 nats is a
# small-but-real next-pitch sharpening; below it we call selection "flat beyond count/pitcher".
SEL_MATERIAL = 3e-3

if selection_gain > SEL_MATERIAL:
    sel_branch = "S1"
else:
    sel_branch = "S2"

# --- Delta_order axis, read under D21 ---
# The optimism-bias scale of min[U,L1]. For WS1's pooling tables this is ~0 on the null world;
# we keep a small positive scale as the boundary between "consistent with null" and
# "over-fragmentation".
D21_BIAS_SCALE = 5e-3
if do_ci["lo"] > 0:
    do_branch = "R1"
elif delta_order > -D21_BIAS_SCALE:
    do_branch = "R2"
else:
    do_branch = "R3"

print("=" * 68)
print(" WS1 RESULTS — branch selector")
print("=" * 68)
print(f" world / mode        : {truth.get('world', DATA_MODE)}")
print(f" C selection loss     : {C_loss:.4f}")
print(f" best history view    : {best_hist_view} at {best_hist:.4f}")
print(f" selection gain (C - best_hist) : {selection_gain:+.4f}  "
      f"(material threshold {SEL_MATERIAL:+.4f})")
print(f" Delta_order          : {delta_order:+.5f}  CI[{do_ci['lo']:+.5f}, {do_ci['hi']:+.5f}]")
print("-" * 68)
print(f" SELECTION BRANCH     : {sel_branch}")
print(f" DELTA_ORDER BRANCH   : {do_branch}")
print("=" * 68)
print(" -> read the matching branch write-ups in the markdown below.")

### Selection branch S1 — *history views clearly beat C*

The best history view lowers next-pitch log loss below the context-only **C** view by more than
noise: **ordered selection structure is present** (finding #1). Prior pitches genuinely help
predict *what is thrown next*. This is the *expected* result on real data — pitchers do sequence
their selection. For the project it means the selection channel is live and worth the fancier
models: WS2's variable-order grammar and WS3's behavior model should extend this edge with
properly calibrated, higher-order dependence. It does **not**, by itself, say anything about
*outcomes* (finding #2) — a predictable pitcher is not necessarily an exploitable one; that is
what the outcome and OPE workstreams test.

### Selection branch S2 — *history barely beats C (flat beyond count/pitcher)*

Adding within-PA history does not sharpen next-pitch prediction beyond context, pitcher, and the
count. For a flat table this is unsurprising and often correct: the selection signal that *is*
there (higher-order, subtle) is below what naive cells can resolve, so empirical Bayes pools it
away. Do **not** over-read this as "pitchers don't sequence" — read it as "a lookup table can't
see it." The proper test of whether a *better* model can is exactly WS2/WS3; if they, too, land
here, the motif literature's modest-selection-structure story is corroborated.

### Δ_order branch R1 — *significantly positive (CI lower bound > 0)*

The fully ordered view beats the better of U/L1 by more than the clustered CI — a **table-visible
ordered dependence**. This is the strong claim, so treat it with suspicion before believing it:
(1) check the **support exhibit** (§7) — if O's edge rides on cells with `n < 20`, it may be
overfitting thin history keys rather than finding real order; (2) check **per-slice consistency**
(§6) — a real ordered effect should concentrate in `long_pa` / `two_strike`, not appear only in
aggregate. If it survives both, WS1 has found genuine ordered structure at the table level, and
WS2/WS3 should confirm and sharpen it. For the project's headline question, a robust R1 is the
first rung of real evidence that *order* — not just history — carries signal.

### Δ_order branch R2 — *≈ 0 or small negative (consistent with no ordering effect)*

Under the D21 rule this reads as **consistent with no table-visible ordering effect** — and,
crucially, **not** as proof that order is absent. Two reasons to withhold the stronger claim.
First, `min[U, L1]` is optimistically biased, so a small negative is the bias, not a cost of
order. Second — and this is the measured caveat — a family table is **mechanism-blind**: on the
positive world it recovered only ~**3%** (attenuation ≈ 0.03) of a real planted effect, because
it sees a velocity mechanism only through the pitch-name → velocity-band proxy. So "≈ 0 from a
table" is fully compatible with "a real but modest ordered *outcome* effect that only a
feature-based or learned model can resolve." This is the honest, expected WS1 result, and it is
precisely the argument for continuing up the ladder: WS3 (features) and WS6 (learned
representation) are built to see what the table cannot. For the project's headline, R2 says *the
transparent baseline finds no order — so any later claim of an ordering effect must clear this
bar and explain why the table missed it.*

### Δ_order branch R3 — *clearly negative, beyond the D21 bias scale*

A negative `Δ_order` larger than the optimism-bias scale is, for WS1's pooling tables,
**diagnostic of over-fragmentation**: the O key has splintered support so badly that its
predictions are noisier than U/L1 even after shrinkage. Diagnose it directly with §7 — expect
elevated `n < 5 / 20` fractions and a short deepest-level backoff bar for O. The fix is not in
WS1 (a flat table has no better move than to pool harder); it is the motivation for WS2's
variable-order backoff, which spends resolution only where support justifies it. For the project,
R3 is a statement about *estimation*, not about baseball: order might still matter, but you
cannot see it by counting cells this fine.

## 10. Discussion and limitations

**What a table can and cannot conclude.** WS1 can say, transparently and with calibrated
uncertainty, how much the *next pitch* and its *expected reward* depend on context, pitcher, and
a shallow slice of within-PA history — and it can quantify, in evaluation exposure, exactly how
the support runs out as the key deepens. What it *cannot* do is resolve subtle or higher-order
ordered dependence (its cells fragment first), represent matchup memory (OM is infeasible), or
see a physical mechanism except through the coarse pitch-name proxy (the ~3% attenuation lesson).
Those are not bugs to fix in WS1; they are the reasons the ladder continues.

**Inheritance — what the later rungs fix.**

- **WS2 (Bayesian variable-order Markov)** keeps WS1's shrinkage philosophy but does backoff
  *properly over ordered tokens*: a variable order that grows only where support allows, with
  calibrated uncertainty. It is the direct answer to WS1's over-fragmentation ceiling (branch
  R3) and its shallow, fixed last-two-family key.
- **WS3 (GBDT stack, the centerpiece)** replaces cells with **features**, so a velocity
  transition is an input in its own right rather than something proxied through family names —
  the direct answer to the mechanism-blindness caveat (branch R2). It also supplies the behavior
  propensities and outcome model the prescriptive workstreams consume.
- **WS4–WS7** move from prediction to prescription and honest counterfactual evaluation, which a
  descriptive table cannot attempt at all.

**The support problem is the through-line.** If there is one figure to carry out of this
notebook, it is §7(b): a third to a half of the evaluation set landing in thin cells once the key
deepens. That single fact — that naive conditioning runs out of data long before it runs out of
questions — is why the project is a *ladder of models* and not a single table. WS1's job is to
make that unavoidable, and to be the honest baseline every later rung must beat.

## 11. Reproducibility appendix

### Exact Phase-2 commands (RUNBOOK Step WS1.1)

The real-data run is driven by the WS1 CLI, on the train seasons (2021–2023) with validation on
2024:

```powershell
conda activate statcast; cd ~\pitch-sequencing-research
python workstreams/ws1_eb_tables/run_ws1.py --table data/processed/decision_table.parquet --out results/ws1/ --views C U L1 O --target both
```

The synthetic CI equivalents (no data needed, ~5 s each):

```powershell
python workstreams/ws1_eb_tables/run_ws1.py --synth null --out results/ws1_null/
python workstreams/ws1_eb_tables/run_ws1.py --synth positive --out results/ws1_pos/
```

If the clustered-bootstrap CIs are slow on the full validation season, lower `--n-boot` (default
200) — it changes only the CI widths, not the point estimates.

### Output files (under `--out`, gitignored)

- `predictions_real_<view>.parquet` — standard-schema predictions per view (C/U/L1/O).
- `ws1_report_real.json` — the full report: per-view losses, baseline references, `Delta_order`
  + CI, support diagnostics, fitted α/κ per level.
- `support_real.csv` — the support-diagnostics exhibit (the table behind §7).
- `ws1_real.runmeta.json` — run metadata: wall-clock seconds, peak RAM, row counts, package
  versions — the raw material for the SPEC §7 performance-vs-compute Pareto plot.

### Run-metadata fields

Each `*.runmeta.json` sidecar records `seconds`, `peak_mem_mb`, `start_mem_mb`, `started_at`,
`n_train`, `n_val`, `views`, and the step tag — one implementation (`pitchseq.runmeta`) used by
every runnable step so the compute accounting is uniform across the study.

### Package versions (this environment)

In [ ]:
import scipy
print("python     :", platform.python_version())
print("numpy      :", np.__version__)
print("pandas     :", pd.__version__)
print("scipy      :", scipy.__version__)
print("matplotlib :", matplotlib.__version__)
print("seed       :", SEED)
print("data mode  :", DATA_MODE)
print("n_boot     :", N_BOOT)